# Resume Bullet Improver

## 1 — Project Overview

This notebook builds an **end-to-end Resume Bullet Improver** that rewrites weak resume bullets into stronger, impact-driven bullets.

What it does:
- Ingests sample resume bullets
- Applies preprocessing and quality checks
- Uses prompt strategies (or robust local fallback) to rewrite bullets
- Produces before/after comparisons
- Applies an optional scoring rubric
- Lets you customize tone and target role

This is built as a teaching notebook: concepts first, then implementation, then evaluation and exercises.

## 2 — Learning Goals

By the end, you will be able to:
1. Explain what makes a resume bullet strong vs weak
2. Design prompt templates for different rewrite strategies
3. Control output tone and target role
4. Compare baseline vs structured rewrite approaches
5. Evaluate rewritten bullets using a practical scoring rubric
6. Identify common mistakes in prompt-based rewriting workflows

## 3 — Problem Statement

Weak resume bullets often:
- Focus on tasks instead of outcomes
- Lack measurable impact
- Use passive or vague language
- Fail to align with a target role

Goal: transform low-signal bullets into **clear, action-oriented, impact-driven** statements while preserving truthfulness.

## 4 — Why This Project Matters

Recruiters scan resumes quickly. Better bullets improve signal density and interview conversion.

Practical GenAI engineering patterns shown here:
- Template-driven prompting
- Structured output validation
- Fallback logic when external LLM service is unavailable
- Rubric-based quality evaluation
- Human-in-the-loop editing workflow

## 5 — Dataset Overview

This project uses a **manually created sample dataset** of weak resume bullets (no external dataset required).

Assumption:
- Examples are synthetic but realistic and safe for learning.
- In production, use your own real bullets and role-specific job descriptions.

## 6 — Environment / Setup

This notebook runs in two modes:
- **LLM mode** (optional): uses local Ollama if available
- **Fallback mode** (default-safe): deterministic rule-based rewriting

This guarantees the notebook remains runnable end-to-end even without external services.

In [ ]:
import json
import re
import urllib.request
import urllib.error
from dataclasses import dataclass
from typing import Dict, List, Any

import pandas as pd


def is_ollama_available(host: str = "http://localhost:11434") -> bool:
    try:
        with urllib.request.urlopen(f"{host}/api/tags", timeout=2) as resp:
            return resp.status == 200
    except Exception:
        return False


OLLAMA_HOST = "http://localhost:11434"
OLLAMA_MODEL = "qwen3:8b"
USE_OLLAMA = is_ollama_available(OLLAMA_HOST)

print(f"Ollama available: {USE_OLLAMA}")
if not USE_OLLAMA:
    print("Running in fallback mode (no external service required).")

## 7 — Imports

Imports are intentionally minimal: standard library + pandas for tabular comparison.

## 8 — Data Loading

We load sample weak bullets with metadata:
- `role_type`: intended target role
- `tone`: preferred writing style
- `weak_bullet`: source text to improve

In [ ]:
examples: List[Dict[str, str]] = [
    {
        "role_type": "Data Analyst",
        "tone": "professional",
        "weak_bullet": "Responsible for creating weekly reports for sales team."
    },
    {
        "role_type": "Software Engineer",
        "tone": "confident",
        "weak_bullet": "Worked on API performance improvements."
    },
    {
        "role_type": "Marketing Manager",
        "tone": "results-driven",
        "weak_bullet": "Helped with social media campaigns and content."
    },
    {
        "role_type": "Operations Associate",
        "tone": "concise",
        "weak_bullet": "Handled inventory and coordinated with vendors."
    },
    {
        "role_type": "Customer Success Manager",
        "tone": "empathetic",
        "weak_bullet": "Answered customer questions and resolved issues."
    },
]

raw_df = pd.DataFrame(examples)
raw_df

## 9 — Data Inspection / EDA

We inspect:
- bullet length distribution
- action-verb presence
- measurable metric presence (numbers, percentages)

In [ ]:
ACTION_VERBS = {
    "built", "designed", "improved", "reduced", "increased", "launched",
    "automated", "optimized", "implemented", "led", "delivered", "created"
}


def has_metric(text: str) -> bool:
    return bool(re.search(r"\b\d+([.,]\d+)?%?\b", text))


def has_action_verb(text: str) -> bool:
    tokens = re.findall(r"[a-zA-Z]+", text.lower())
    return any(t in ACTION_VERBS for t in tokens)


eda_df = raw_df.copy()
eda_df["word_count"] = eda_df["weak_bullet"].str.split().str.len()
eda_df["has_metric"] = eda_df["weak_bullet"].apply(has_metric)
eda_df["has_action_verb"] = eda_df["weak_bullet"].apply(has_action_verb)

print("EDA summary:")
print(eda_df[["role_type", "word_count", "has_metric", "has_action_verb"]])
print("\nAverage word count:", round(eda_df["word_count"].mean(), 2))

## 10 — Data Preprocessing

We clean bullets before rewriting:
- strip extra whitespace
- normalize punctuation spacing
- preserve original semantics (no fabricated claims)

In [ ]:
def clean_bullet(text: str) -> str:
    t = re.sub(r"\s+", " ", text.strip())
    t = re.sub(r"\s+([.,;:!?])", r"\1", t)
    return t


prep_df = raw_df.copy()
prep_df["clean_bullet"] = prep_df["weak_bullet"].apply(clean_bullet)
prep_df[["weak_bullet", "clean_bullet"]]

## 11 — Baseline Approach

Baseline = simple rule-based template rewrite (no LLM):
- replace weak starters like "Responsible for" / "Worked on"
- force action-oriented start

This is intentionally basic so we can compare it against the main workflow.

In [ ]:
def baseline_rewrite(text: str) -> str:
    t = clean_bullet(text)
    t = re.sub(r"^Responsible for\s+", "Created and managed ", t, flags=re.IGNORECASE)
    t = re.sub(r"^Worked on\s+", "Improved ", t, flags=re.IGNORECASE)
    t = re.sub(r"^Helped with\s+", "Contributed to ", t, flags=re.IGNORECASE)
    t = re.sub(r"^Handled\s+", "Managed ", t, flags=re.IGNORECASE)
    t = re.sub(r"^Answered\s+", "Resolved ", t, flags=re.IGNORECASE)
    if not t.endswith("."):
        t += "."
    return t


prep_df["baseline_bullet"] = prep_df["clean_bullet"].apply(baseline_rewrite)
prep_df[["clean_bullet", "baseline_bullet"]]

## 12 — Main Model / Workflow

### Prompt Strategies
We implement three practical prompt strategies:
1. **Impact-first**: prioritize measurable outcome
2. **STAR-lite**: action + context + result
3. **Role-tailored**: align wording to the target role

### Structured Output
We request strict JSON:
```json
{ "improved_bullet": "...", "reasoning": "...", "missing_metric_hint": "..." }
```

If LLM is unavailable, we return a deterministic structured fallback with the same schema.

In [ ]:
def build_prompt(weak_bullet: str, role_type: str, tone: str, strategy: str) -> str:
    return f"""
You are a resume writing assistant.
Rewrite the weak resume bullet into one strong, truthful, impact-driven bullet.

Constraints:
- Do NOT invent fake numbers or claims.
- Keep it to one bullet sentence.
- Tone: {tone}
- Target role: {role_type}
- Strategy: {strategy}

Weak bullet:
{weak_bullet}

Return valid JSON with keys:
- improved_bullet
- reasoning
- missing_metric_hint
""".strip()


def call_ollama_json(prompt: str, host: str = OLLAMA_HOST, model: str = OLLAMA_MODEL) -> Dict[str, Any]:
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "format": "json",
    }
    req = urllib.request.Request(
        f"{host}/api/generate",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read().decode("utf-8"))

    response_text = data.get("response", "{}")
    try:
        parsed = json.loads(response_text)
    except json.JSONDecodeError:
        start = prompt.find("Weak bullet:")
        end = prompt.find("Return valid JSON")
        if start != -1 and end != -1 and end > start:
            weak_part = prompt[start + len("Weak bullet:"):end].strip()
        else:
            weak_part = "Worked on key initiatives."

        parsed = {
            "improved_bullet": baseline_rewrite(weak_part),
            "reasoning": "Fallback parse because model output was not valid JSON.",
            "missing_metric_hint": "Add a measurable result if available (%, $, time saved, volume).",
        }

    return parsed


def fallback_structured_rewrite(weak_bullet: str, role_type: str, tone: str, strategy: str) -> Dict[str, str]:
    base = baseline_rewrite(weak_bullet)
    if strategy == "impact_first":
        improved = f"{base[:-1]} to improve team outcomes in {role_type.lower()} workflows."
    elif strategy == "star_lite":
        improved = f"{base[:-1]}, supporting measurable delivery for {role_type.lower()} goals."
    else:
        improved = f"{base[:-1]}, tailored to {role_type.lower()} priorities with a {tone} tone."

    return {
        "improved_bullet": improved,
        "reasoning": f"Applied {strategy} strategy with {tone} tone for {role_type}.",
        "missing_metric_hint": "Add a concrete metric (%, count, revenue, time) if you can verify one.",
    }


def rewrite_bullet(weak_bullet: str, role_type: str, tone: str, strategy: str = "impact_first") -> Dict[str, str]:
    if USE_OLLAMA:
        prompt = build_prompt(weak_bullet, role_type, tone, strategy)
        return call_ollama_json(prompt)
    return fallback_structured_rewrite(weak_bullet, role_type, tone, strategy)


## 13 — Inference / Pipeline Execution

We run three strategies per bullet and keep one chosen output for final comparison.

In [ ]:
records: List[Dict[str, Any]] = []

for item in examples:
    weak = item["weak_bullet"]
    role_type = item["role_type"]
    tone = item["tone"]

    out_impact = rewrite_bullet(weak, role_type, tone, strategy="impact_first")
    out_star = rewrite_bullet(weak, role_type, tone, strategy="star_lite")
    out_role = rewrite_bullet(weak, role_type, tone, strategy="role_tailored")

    chosen = out_impact.get("improved_bullet", baseline_rewrite(weak))

    records.append({
        "role_type": role_type,
        "tone": tone,
        "before": weak,
        "after_baseline": baseline_rewrite(weak),
        "after_impact_first": out_impact.get("improved_bullet", ""),
        "after_star_lite": out_star.get("improved_bullet", ""),
        "after_role_tailored": out_role.get("improved_bullet", ""),
        "chosen_after": chosen,
        "hint": out_impact.get("missing_metric_hint", "")
    })

result_df = pd.DataFrame(records)
result_df[["role_type", "before", "chosen_after"]]

## 14 — Evaluation and Interpretation

We use an **optional scoring rubric** (0 to 10):
- Action orientation (0-3)
- Outcome/impact signal (0-3)
- Clarity and concision (0-2)
- Role/tone alignment (0-2)

This is a practical heuristic, not a perfect metric.

In [ ]:
def score_bullet(text: str, role_type: str, tone: str) -> Dict[str, float]:
    t = text.lower()

    action = 3.0 if has_action_verb(text) else 1.0
    impact = 3.0 if has_metric(text) else (2.0 if "improve" in t or "outcome" in t or "result" in t else 1.0)

    wc = len(text.split())
    clarity = 2.0 if 10 <= wc <= 28 else 1.0

    role_hint = role_type.split()[0].lower()
    tone_hit = 1.0 if tone.lower() in t else 0.5
    align = 2.0 if (role_hint in t and tone_hit >= 1.0) else 1.0

    total = action + impact + clarity + align
    return {
        "action": action,
        "impact": impact,
        "clarity": clarity,
        "alignment": align,
        "total": round(total, 2)
    }


before_scores = []
after_scores = []
for _, row in result_df.iterrows():
    before_scores.append(score_bullet(row["before"], row["role_type"], row["tone"])["total"])
    after_scores.append(score_bullet(row["chosen_after"], row["role_type"], row["tone"])["total"])

result_df["score_before"] = before_scores
result_df["score_after"] = after_scores
result_df["score_gain"] = result_df["score_after"] - result_df["score_before"]

print("Average score before:", round(result_df["score_before"].mean(), 2))
print("Average score after:", round(result_df["score_after"].mean(), 2))
print("Average gain:", round(result_df["score_gain"].mean(), 2))

result_df[["role_type", "score_before", "score_after", "score_gain"]]

### Before/After Comparison

Review the transformations side by side.

In [ ]:
comparison_cols = ["role_type", "before", "chosen_after", "hint"]
result_df[comparison_cols]

## 15 — Error Analysis / Limitations

Known limitations:
- If source bullets lack measurable context, outputs remain generic
- Heuristic scoring is approximate and can be gamed
- LLM outputs can vary by model/version
- Tone control may overfit style words instead of improving substance

Assumption note:
- This notebook avoids fabricating metrics; it suggests metric placeholders instead.

## 16 — How to Improve It

### Production Improvement Ideas
1. Add a domain-specific verb and impact lexicon by role family (engineering, sales, ops, product)
2. Add retrieval from user achievements (projects, KPIs) to inject truthful metrics
3. Add structured guardrails (JSON schema + semantic validators)
4. Add human review UI with accept/edit/reject workflow
5. Add offline eval set with recruiter-labeled quality scores
6. Add privacy controls for sensitive resume text

## 17 — Practice Exercises

### Mini Challenge
Pick one weak bullet from your own resume and produce:
- one concise version
- one leadership-focused version
- one metric-focused version

Then score all three with the rubric and justify which is best.

### Customization Exercises
1. Change tone to `executive`, `technical`, and `friendly` and compare outputs
2. Change role type (e.g., Data Analyst → Product Manager) and inspect wording shifts
3. Add a new prompt strategy called `stakeholder_impact`
4. Tighten max length to 22 words and evaluate tradeoffs

### Try This Next
- Add job description keywords and force keyword alignment
- Add an ATS keyword coverage score
- Export final bullets to markdown or JSON for portfolio reuse

## Common Mistakes (Important)

1. **Inventing fake metrics**
   Never fabricate values like "increased revenue by 40%" without evidence.
2. **Task-only bullets**
   "Responsible for..." says duty, not impact.
3. **Overly long bullets**
   Dense bullets reduce readability.
4. **Ignoring target role**
   Strong bullets are role-specific, not generic.
5. **No human review**
   LLM output should be edited by the user before final use.

## 18 — Final Takeaway / Summary

You now have a complete, runnable pipeline for improving resume bullets with:
- preprocessing
- baseline rewrite
- structured strategy-driven rewriting
- before/after comparison
- optional quality scoring rubric

Most important principle: **improve clarity and impact without inventing facts**.

Use this notebook as a practical starting point for a production-grade resume assistant.